# Лабораторная 1. Языковая модель из счетчиков

Луценко Вячеслав Александрович, М092501(77). Очное отделение.

Считаю пары слов в «Войне и мире», генерирую текст и проверяю, что меняет сглаживание. Используется только стандартная библиотека Python. Корпус отдельно, в Git его не добавляю.

In [1]:
from collections import Counter
from itertools import islice
import hashlib
import json
import math
from pathlib import Path
import random
import re
import sys
import time

HERE = Path("lab01") if Path("lab01/bigram.py").is_file() else Path(".")
sys.path.insert(0, str(HERE.resolve()))
from bigram import BigramModel, WORD_RE, UNK, tokenize
from run import SOURCE, SOURCE_BLOB, check_corpus

started = time.perf_counter()
SEED = 42
print("Python:", sys.version.split()[0])
print("Seed:", SEED)

Python: 3.13.0
Seed: 42


## Корпус и разбиение

Знаки препинания выбрасываю: модель предсказывает слова, а не пунктуацию. Слова с дефисами сохраняю, «ё» заменяю на «е». Пары через границы предложений тоже учитываются - это ограничение такого простого разбиения. Последние 10 % слов откладываю до обучения, включая выбор словаря.

In [2]:
raw = (HERE / "corpus.txt").read_bytes()
check_corpus(raw)
text = raw.decode("utf-8").replace("\r\n", "\n")
normalized = text.lower().replace("ё", "е")
words = tokenize(text)
assert len(words) >= 100_000
cut = int(len(words) * 0.9)
train_words, val_words = words[:cut], words[cut:]
counts = Counter(words)
train_vocabulary = set(train_words)
oov = sum(word not in train_vocabulary for word in val_words)

statistics = {
    "words": len(words),
    "vocab_size_corpus": len(counts),
    "train_words": len(train_words),
    "val_words": len(val_words),
    "train_vocabulary": len(train_vocabulary),
    "val_oov_words": oov,
    "val_oov_fraction": oov / len(val_words),
    "top20": counts.most_common(20),
}
print("Источник:", SOURCE)
print("SHA-256:", hashlib.sha256(raw).hexdigest())
print("Всего слов:", len(words), "| Словарь корпуса:", len(counts))
print("Обучение:", len(train_words), "| Проверка:", len(val_words))
print("Словарь обучения:", len(train_vocabulary), "+ один <unk>")
print(f"Неизвестных слов в проверке: {oov} ({oov / len(val_words):.2%})")
print("20 частых слов:", counts.most_common(20))

Источник:

 https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/WarAndPeace.txt
SHA-256: 5f088a66e1f3992153555fdf0361652f759697e429bd698fb2900fdf26df9836
Всего слов: 433082 | Словарь корпуса: 48117
Обучение: 389773 | Проверка: 43309
Словарь обучения: 45309 + один <unk>
Неизвестных слов в проверке: 3496 (8.07%)
20 частых слов: [('и', 21085), ('в', 10870), ('не', 8520), ('что', 7600), ('он', 7395), ('на', 6665), ('с', 5861), ('как', 4011), ('его', 3895), ('к', 3428), ('я', 2854), ('она', 2732), ('но', 2725), ('все', 2674), ('было', 2501), ('это', 2434), ('то', 2261), ('сказал', 2000), ('так', 1954), ('о', 1832)]


## Счетчики и вероятности

Класс находится в `bigram.py`: `fit`, `prob`, `generate`, `logprob`. В знаменателе число исходящих пар из слова. Это почти его частота, но последнее слово обучающей части не образует перехода через границу проверки. Неизвестные слова отображаются в один заранее добавленный `<unk>`.

In [3]:
fit_started = time.perf_counter()
model = BigramModel().fit(train_words)
fit_seconds = time.perf_counter() - fit_started
print(f"Обучение счетчиков: {fit_seconds:.3f} с")
print("Разных пар:", sum(len(row) for row in model.next_counts.values()))
print("Слово «князь»: частота =", model.word_counts["князь"],
      ", исходящих переходов =", model.context_counts["князь"])
print("Продолжения:", model.next_counts["князь"].most_common(10))
print("P(андрей | князь) =", round(model.prob("князь", "андрей"), 6))
print("Сумма строки =", sum(model.prob("князь", w) for w in model.next_counts["князь"]))

Обучение счетчиков: 0.371 с
Разных пар: 236820
Слово «князь»: частота = 1321 , исходящих переходов = 1321
Продолжения: [('андрей', 775), ('василий', 147), ('багратион', 36), ('николай', 32), ('ипполит', 23), ('и', 17), ('не', 13), ('в', 8), ('был', 8), ('болконский', 7)]
P(андрей | князь) = 0.586677
Сумма строки = 1.0


## Пять генераций по 30 слов

Каждый раз выбираю следующее слово случайно по строке вероятностей. Разные старты и фиксированные seed позволяют повторить примеры.

In [4]:
generations = []
for number, start in enumerate(("князь", "наташа", "пьер", "война", "он")):
    generated = model.generate(start, n_words=30, seed=SEED + number)
    assert len(generated.split()) == 30
    generations.append({"start": start, "seed": SEED + number, "text": generated})
    print(f"{number + 1}. {generated}\n")

1. князь андрей не правда ли еще лаврушка кричал он очевидно не могу знать что он невольно этим штабным адъютантикам к ее ожидал найти кого-нибудь из виду цепи и что безухов

2. наташа не говори да это какой мере надо сходить к и чувствуя всю правду выражалось одно и неловкость продолжалась лакей входя в князе андрее она продолжала целовать руки и он

3. пьер остановившись на москву он может быть и с своею оживленностью какое-то духовное лицо к самому поверьте что эти ночные посещения правда что задремавшая графиня просить для себя ростов был

4. война какая прелесть ее отцом и с своими подданными запишет здорово и оттуда старшая катишь ты говоришь наша сердце красневшей и прилежным очищением самого чудака его влекло его в голубых

5. он ухаживал за чаем довезти до земли генералитет последовал за что там пьер подъезжал тем тяжеле снопа ржаного завтра отвечал тихон слушала обратился к красивой и начал говорить с свитским



## Температура

Вероятности возводятся в степень 1/T и снова нормируются. Для сравнения оставляю один старт и один seed. Кроме текста смотрю энтропию распределения после «князь»: она измеряет разнообразие выбора.

In [5]:
temperature_results = []
row = model.next_counts["князь"]
for temperature in (0.5, 1.0, 2.0):
    powered = [value ** (1 / temperature) for value in row.values()]
    probabilities = [value / sum(powered) for value in powered]
    entropy = -math.fsum(p * math.log(p) for p in probabilities)
    generated = model.generate("князь", n_words=30, temperature=temperature, seed=SEED)
    temperature_results.append({
        "temperature": temperature, "entropy_nats": entropy,
        "largest_probability": max(probabilities), "text": generated,
    })
    print(f"T={temperature}: энтропия {entropy:.3f}, самая вероятная пара {max(probabilities):.3f}")
    print(generated, "\n")

T=0.5: энтропия 0.204, самая вероятная пара 0.959
князь андрей не было он с тем чтобы не было в москве и с ним разговор и не мог удержаться чтобы в темноте ночи не говорили что он не был 

T=1.0: энтропия 2.293, самая вероятная пара 0.587
князь андрей не правда ли еще лаврушка кричал он очевидно не могу знать что он невольно этим штабным адъютантикам к ее ожидал найти кого-нибудь из виду цепи и что безухов 

T=2.0: энтропия 4.899, самая вероятная пара 0.100
князь быть была все молчало крестилось только помнила этот возьмет ее лоб широкими скулами красивым принялись с принятием таинства брака будь здоров и дальше несмотря ни той губернии он выразил 



## Связное предложение и перестановка

Беру первое полное предложение на 12-25 русских слов после границы проверки. Этот критерий задан заранее. Перестановка фиксирована seed=42; слова и длина в обоих вариантах одинаковые.

In [6]:
val_start = next(islice(WORD_RE.finditer(normalized), cut, None)).start()
val_text = normalized[val_start:]
sentence_words = next(
    tokenize(fragment)
    for fragment in re.split(r"[.!?…]+", val_text)[1:]
    if 12 <= len(tokenize(fragment)) <= 25
)
shuffled_words = sentence_words.copy()
random.Random(SEED).shuffle(shuffled_words)
plain_logprob = model.logprob(sentence_words)
shuffled_logprob = model.logprob(shuffled_words)
comparison = {
    "sentence": " ".join(sentence_words),
    "shuffled": " ".join(shuffled_words),
    "plain_without_smoothing": plain_logprob if math.isfinite(plain_logprob) else "-inf",
    "shuffled_without_smoothing": shuffled_logprob if math.isfinite(shuffled_logprob) else "-inf",
}
print("Связное:", comparison["sentence"])
print("Перестановка:", comparison["shuffled"])
print("Без сглаживания: log P =", plain_logprob, "и", shuffled_logprob)

Связное: на другой день были у фельдмаршала обед и бал которые государь удостоил своим присутствием
Перестановка: бал своим и обед присутствием удостоил фельдмаршала день которые были у на другой государь
Без сглаживания: log P = -inf и -inf


Если хотя бы одной пары не было, ее вероятность равна нулю, а логарифм равен минус бесконечности. Когда обе суммы равны минус бесконечности, ранжировать предложения по ним нельзя. Добавляю α к каждой ячейке строки, включая `<unk>`, и αV к знаменателю. Неизвестный контекст при таком сглаживании дает равномерное распределение.

In [7]:
smoothing_results = []
for alpha in (1.0, 0.1, 0.01):
    model.alpha = alpha  # Меняется только сглаживание, счетчики не переобучаются.
    coherent_lp = model.logprob(sentence_words)
    mixed_lp = model.logprob(shuffled_words)
    perplexity = model.perplexity(val_words)
    assert math.isfinite(perplexity)
    smoothing_results.append({
        "alpha": alpha, "perplexity": perplexity,
        "sentence_logprob": coherent_lp, "shuffled_logprob": mixed_lp,
    })
    print(f"alpha={alpha:<4}: perplexity={perplexity:.3f}; "
          f"log P связного={coherent_lp:.3f}, перестановки={mixed_lp:.3f}")
model.alpha = 0.0
best = min(smoothing_results, key=lambda result: result["perplexity"])
print("Лучшее значение на отложенной части:", best["alpha"])
print("Число оцененных переходов:", len(val_words) - 1)

alpha=1.0 : perplexity=15748.113; log P связного=-124.387, перестановки=-129.288
alpha=0.1 : perplexity=7743.544; log P связного=-111.136, перестановки=-123.554


alpha=0.01: perplexity=5309.636; log P связного=-101.409, перестановки=-122.487
Лучшее значение на отложенной части: 0.01
Число оцененных переходов: 43308


## Сохранение результатов

Перплексия делится на число оцененных переходов, то есть на число слов минус один. Первое слово не оцениваю, так как для него нет контекста. Лучшее α выбрано по отложенной части: отдельной независимой тестовой выборки в этой работе нет.

In [8]:
results = {
    "source": SOURCE,
    "source_git_blob": SOURCE_BLOB,
    "source_sha256": hashlib.sha256(raw).hexdigest(),
    "seed": SEED,
    "statistics": statistics,
    "fit_seconds": fit_seconds,
    "generations": generations,
    "temperatures": temperature_results,
    "comparison": comparison,
    "smoothing": smoothing_results,
    "evaluated_transitions": len(val_words) - 1,
    "python": sys.version.split()[0],
    "experiment_seconds": time.perf_counter() - started,
}
artifacts = HERE / "artifacts"
artifacts.mkdir(exist_ok=True)
(artifacts / "results.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
print("Сохранено:", artifacts / "results.json")
print(f"Полный эксперимент: {results['experiment_seconds']:.3f} с")

Сохранено: artifacts\results.json
Полный эксперимент: 0.876 с


## Что здесь не умеет модель

Контекст состоит из одного слова. Модель может сохранять частые соседства, но не отслеживает сюжет, согласование на расстоянии и границы предложений. При T=0.5 чаще выбираются привычные продолжения, при T=2 распределение становится ровнее. Слишком большое α раздает много массы редким и невстречавшимся парам: V велико, поэтому слагаемое αV быстро перевешивает реальные счетчики.